In [149]:
# Обновление pip
!python -m pip install --upgrade pip

!pip install torch==2.8.0+cu126 --extra-index-url https://download.pytorch.org/whl/cu126
!pip install cuml-cu12 catboost optuna lightgbm --quiet

Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu126
  Using cached https://download.pytorch.org/whl/cu126/nvidia_cuda_nvrtc_cu12-12.6.77-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached https://download.pytorch.org/whl/cu126/nvidia_cublas_cu12-12.6.4.1-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.5 kB)
  Using cached https://download.pytorch.org/whl/cu126/nvidia_cufft_cu12-11.3.0.4-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.5 kB)
  Using cached https://download.pytorch.org/whl/cu126/nvidia_curand_cu12-10.3.7.77-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.5 kB)
  Using cached https://download.pytorch.org/whl/cu126/nvidia_cusolver_cu12-11.7.1.2-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.6 kB)
  Using cached https://download.pytorch.org/whl/cu126/nvidia_cusparse_cu12-12.5.4.2-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.6

In [150]:

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    make_scorer
)

from sklearn.model_selection import (
    StratifiedKFold,
    KFold,
    cross_val_score
)

from collections import Counter

from sklearn.decomposition import PCA

from sklearn.preprocessing import (
    PolynomialFeatures,
    StandardScaler,
    OneHotEncoder,
    LabelEncoder
)
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    AdaBoostRegressor,
)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor

import warnings
warnings.filterwarnings("ignore")
%matplotlib inline

sns.set_theme()

import optuna

from google.colab import drive
drive.mount('/content/gdrive')

ImportError: cannot import name 'output_can_be_silenced' from 'IPython.core.magic' (/usr/local/lib/python3.12/dist-packages/IPython/core/magic.py)

In [157]:
cd "/content/gdrive/MyDrive/nto/SECOND STEP"

/content/gdrive/MyDrive/nto/SECOND STEP


In [192]:
!git add -A
!git commit -m "CBR-V1-adding"

[main b668f6a] CBR-V1-adding
 1 file changed, 1 insertion(+), 1 deletion(-)
 rewrite main.ipynb (72%)


In [193]:
!git push --set-upstream origin main

Enumerating objects: 33, done.
Counting objects: 100% (33/33), done.
Delta compression using up to 2 threads
Compressing objects: 100% (32/32), done.
Writing objects: 100% (32/32), 22.89 MiB | 2.57 MiB/s, done.
Total 32 (delta 8), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (8/8), done.
remote: warning: See https://gh.io/lfs for more information.
remote: warning: File data/book_descriptions.csv is 57.10 MB; this is larger than GitHub's recommended maximum file size of 50.00 MB
remote: warning: GH001: Large files detected. You may want to try Git Large File Storage - https://git-lfs.github.com.
To https://github.com/chisem29/NTO-AI.git
   2c43906..b668f6a  main -> main
Branch 'main' set up to track remote branch 'main' from 'origin'.


In [187]:
!git pull origin main

From https://github.com/chisem29/NTO-AI
 * branch            main       -> FETCH_HEAD
Already up to date.


In [177]:
!git branch

* main


In [166]:
!git config --global user.email tonyfhg1000@gmail.com
!git config --global user.name chisem29

In [148]:
!pip show torch
!pip show cuml-cu12

Name: torch
Version: 2.8.0+cu126
Summary: Tensors and Dynamic neural networks in Python with strong GPU acceleration
Home-page: https://pytorch.org/
Author: PyTorch Team
Author-email: packages@pytorch.org
License: BSD-3-Clause
Location: /usr/local/lib/python3.12/dist-packages
Requires: filelock, fsspec, jinja2, networkx, nvidia-cublas-cu12, nvidia-cuda-cupti-cu12, nvidia-cuda-nvrtc-cu12, nvidia-cuda-runtime-cu12, nvidia-cudnn-cu12, nvidia-cufft-cu12, nvidia-cufile-cu12, nvidia-curand-cu12, nvidia-cusolver-cu12, nvidia-cusparse-cu12, nvidia-cusparselt-cu12, nvidia-nccl-cu12, nvidia-nvjitlink-cu12, nvidia-nvtx-cu12, setuptools, sympy, triton, typing-extensions
Required-by: accelerate, fastai, peft, sentence-transformers, timm, torchaudio, torchdata, torchvision
Name: cuml-cu12
Version: 25.10.0
Summary: cuML - RAPIDS ML Algorithms
Home-page: https://github.com/rapidsai/cuml
Author: NVIDIA Corporation
Author-email: 
License: Apache-2.0
Location: /usr/local/lib/python3.12/dist-packages
Requ

In [7]:
STAT = dict(
    users='/content/gdrive/MyDrive/nto/SECOND STEP/data/users.csv',
    book_genres='/content/gdrive/MyDrive/nto/SECOND STEP/data/book_genres.csv',
    books='/content/gdrive/MyDrive/nto/SECOND STEP/data/books.csv',
    genres='/content/gdrive/MyDrive/nto/SECOND STEP/data/genres.csv',
    book_descriptions='/content/gdrive/MyDrive/nto/SECOND STEP/data/book_descriptions.csv'
)

TRAIN_PATH = '/content/gdrive/MyDrive/nto/SECOND STEP/data/train.csv',
TEST_PATH = '/content/gdrive/MyDrive/nto/SECOND STEP/data/test.csv'

In [8]:
data_test = pd.read_csv(TEST_PATH, sep=',')

data_test['user_id'] = data_test['user_id'].astype('str')
data_test['book_id'] = data_test['book_id'].astype('str')

data_test.head()

,user_id,book_id
0,281,2461928
1,1250,31957
2,4241,196603
3,5140,468894
4,7781,2141951


In [9]:
data = pd.concat(map(pd.read_csv, TRAIN_PATH))

data['user_id'] = data['user_id'].astype('str')
data['book_id'] = data['book_id'].astype('str')

data.head()

,user_id,book_id,has_read,rating,timestamp
0,281,441829,1,10,2007-04-11 06:09:42
1,281,168663,1,10,2007-04-11 06:10:12
2,1851,454829,0,0,2007-11-26 22:26:15
3,1851,431081,1,10,2007-11-26 23:25:24
4,1851,309911,0,0,2007-11-28 21:35:29


In [11]:
set1 = set(data_test['user_id'])
set2 = set(data['user_id'])
print(len(set1 & set2))

2894


In [27]:
set1 = set(data_test['user_id'])
filCol = data.groupby('user_id')['has_read']
set2 = set(data.loc[(filCol.transform('sum') > 0) |
                    (filCol.transform('count') >= 3), 'user_id'])
len(set2 & set1)

2894

In [16]:
data.shape

(268581, 5)

In [ ]:
data.isna().sum()

In [ ]:
data[(data['has_read'] == 0) & (data['rating'] > 0)]

In [ ]:
ax = sns.countplot(data=data, x='has_read', hue='has_read', legend=False)
ax.legend([])
for p in ax.patches:
    height = p.get_height()
    ax.text(p.get_x() + p.get_width() / 2., height, int(height),
            ha="center", va="bottom")

In [28]:
data['timestamp'] = pd.to_datetime(data['timestamp'])
data['year'] = data['timestamp'].dt.year
data['month'] = data['timestamp'].dt.month
data['day'] = data['timestamp'].dt.day

In [ ]:
sns.countplot(data=data, x='year', hue='year')

In [29]:
users = pd.read_csv(STAT['users'], sep=',')
users['user_id'] = users['user_id'].astype(str)
users.columns

Index(['user_id', 'gender', 'age'], dtype='object')

In [ ]:
len(users)

7277

In [ ]:
users['age'].describe()

In [ ]:
sns.countplot(data=users, x='gender', hue='gender')

In [30]:
books = pd.read_csv(STAT['books'], sep=',')
books['book_id'] = books['book_id'].astype(str)
books.isna().sum()

,0
book_id,0
title,0
author_id,0
author_name,0
publication_year,0
language,0
publisher,0
avg_rating,455


In [31]:
books['avg_rating'] = books['avg_rating'].fillna(0)

In [32]:
book_genres = pd.read_csv(STAT['book_genres'], sep=',')
book_genres['book_id'] = book_genres['book_id'].astype(str)
book_genres['genre_id'] = book_genres['genre_id'].astype(str)
book_genres.columns

Index(['book_id', 'genre_id'], dtype='object')

In [33]:
genres = pd.read_csv(STAT['genres'], sep=',')
genres['genre_id'] = genres['genre_id'].astype(str)
genres.columns

Index(['genre_id', 'genre_name', 'books_count'], dtype='object')

In [ ]:
len(book_genres['book_id'].unique()), len(book_genres['book_id'])

(50489, 94954)

In [ ]:
book_genres.groupby('book_id')['genre_id'].count().describe()

,genre_id
count,50489.000000
mean,1.880687
std,0.875909
min,1.000000
25%,1.000000
50%,2.000000
75%,2.000000
max,10.000000


In [151]:
def tablePrep(df, train=False) :
  data_with_users = df.merge(users, on='user_id', how='left')
  data_with_ub = data_with_users.merge(books, on='book_id', how='left')
  data_with_ubg = data_with_ub.drop(columns=['author_id']).merge(book_genres, on='book_id', how='left')

  for col in ['title', 'author_name', 'language', 'publisher'] :
    data_with_ubg[col] = data_with_ubg[col].astype('category')

  if train :
    filCol = data_with_ubg.groupby('user_id')['has_read']
    return data_with_ubg.loc[(filCol.transform('sum') > 0) |
                    (filCol.transform('count') >= 3)].drop(
                        columns=['timestamp', 'month', 'year', 'day']
                        )

  data_with_ubg['has_read'] = 0
  return data_with_ubg

In [152]:
data_with_ubg_TN = tablePrep(data, True)
X, Y = data_with_ubg_TN.drop(columns=['rating']), data_with_ubg_TN['rating']

In [105]:
class TuningModel :
  def __init__(
      self,
      model,
      direction=None,
      params={},
      init_params={},
      preprocessing=None
  ) :
    self._model = model
    self.study = optuna.create_study(
         direction=direction
    )
    self.params = params
    self.init_params = init_params

    self.preprocessing = preprocessing
    self._trained_pipe = None

    self.scoresByTrial = {}

  def __getitem__(self, trial) :

    return Pipeline([
        ('Prep', self.preprocessing),
        ('Model', self._model(
            **self.init_params,
            **self.study.trials[trial].params)
        )
    ])

  def objectiveBuild(self,
      X_train,
      Y_train,
      score_params=None,
      n_splits=5,
    ) :

      def objective(trial):

        paramsSuggestions = {}
        paramsSuggestions.update(self.init_params)

        for param in self.params :
          if ('max' in param or \
              'min' in param or \
              'l2' in param or \
              'num' in param or \
              param in (
                  'border_count',
                  "random_strength",
                  'n_estimators')
            ) and param != "max_features" :
              paramsSuggestions[param] = trial.suggest_int(**self.params[param], name=param)
          elif 'learning_rate' in param or \
                'reg' in param or \
                'subsample' in param or \
                param in ("colsample_bytree",
                  'gamma'):
              paramsSuggestions[param] = trial.suggest_float(**self.params[param], name=param)
          else :
            paramsSuggestions[param] = trial.suggest_categorical(choices=self.params[param], name=param)

        pipe = Pipeline([
            ('Prep', self.preprocessing),
            ('Model', self._model(**paramsSuggestions))
        ])

        cv_score = cross_val_score(
            pipe,
            X=X_train,
            y=Y_train,
            cv=StratifiedKFold(n_splits=n_splits),
            scoring=make_scorer(**score_params)
        )

        mean_cv_score = cv_score.mean()
        self.scoresByTrial[len(self.scoresByTrial) + 1] = cv_score

        return mean_cv_score

      return objective

  def train(
      self,
      n_trials,
      X_train,
      Y_train,
      score_params=None,
      n_splits=5,
    ) :

    self.study.optimize(
      self.objectiveBuild(
        X_train,
        Y_train,
        score_params,
        n_splits),
      n_trials=n_trials
    )

    self._trained_pipe = Pipeline([
        ('Prep', self.preprocessing),
        ('Model', self.estimator)
    ])

    self.fit(X_train, Y_train)

  def fit(self, X_train, Y_train) :
    return self._trained_pipe.fit(X_train, Y_train)

  def predict(self, X) :
    return self._trained_pipe.predict(X)

  @property
  def estimator(self) :
    return self._model(
        **self.study.best_params,
        **self.init_params)

In [51]:
def score(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    score = 1 - (0.5 * rmse / 10 + 0.5 * mae / 10)
    return score

In [135]:
CatR2 = TuningModel(
    CatBoostRegressor,
    'maximize',
    params={
      'n_estimators' : {'low' : 75, 'high' : 300, 'step' : 15},
      'learning_rate' : {'low' : 1e-2, 'high' : 95e-2, 'step' : 2e-4},
      'max_depth' : {'low' : 2, 'high' : 5, 'step' : 1},
      'l2_leaf_reg': {'low': 1, 'high': 10, 'step': 1},
      'border_count': {'low': 32, 'high': 255, 'step': 16},
      'random_strength': {'low': 0, 'high': 20, 'step': 2},
      'min_data_in_leaf': {'low': 1, 'high': 50, 'step': 2},
    },
    init_params={
      'logging_level' : 'Silent',
      'task_type' : 'GPU',
      'cat_features' : X.select_dtypes(include=['object', 'category']).columns.tolist()
    },
)

[I 2025-11-13 18:56:31,479] A new study created in memory with name: no-name-380e558e-377e-4a30-9871-f41b778d964b


In [136]:
CatR2.train(
    1,
    X_train=X,
    Y_train=Y,
    score_params={
        "score_func" : score
    },
    n_splits=5
)

[I 2025-11-13 18:57:09,789] Trial 0 finished with value: 0.8134734452958401 and parameters: {'n_estimators': 225, 'learning_rate': 0.1542, 'max_depth': 2, 'l2_leaf_reg': 10, 'border_count': 32, 'random_strength': 4, 'min_data_in_leaf': 39}. Best is trial 0 with value: 0.8134734452958401.


In [156]:
CatR2._trained_pipe.named_steps['Model'].save_model('CBRV1.cbm')

In [137]:
data_with_ubg_T = tablePrep(data_test)[X.columns]

In [138]:
preds = CatR2.predict(data_with_ubg_T)

In [ ]:
len(preds)

2894

In [139]:
df = pd.concat([data_with_ubg_T[['user_id', 'book_id']].reset_index(drop=True),
                pd.DataFrame([preds]).T.reset_index(drop=True)], axis=1)
df = df.rename(columns={0 : 'rating_predict'})
df.head()

,user_id,book_id,rating_predict
0,281,2461928,0.627777
1,1250,31957,0.039401
2,1250,31957,0.039401
3,1250,31957,0.039401
4,4241,196603,0.921844


In [141]:
df.to_csv('submissions.csv', index=False, sep=',')